In [1]:
print("hello")

hello


In [2]:
## Data Ingestion

from langchain_core.documents import Document

In [3]:
doc = Document(
    page_content="this is the main page text context i am using this to create rag",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "minahil",
        "date_created": "30-4-2025"
    }
)



In [4]:
doc

Document(metadata={'source': 'example.txt', 'page': 1, 'author': 'minahil', 'date_created': '30-4-2025'}, page_content='this is the main page text context i am using this to create rag')

In [7]:
import os

# Create folder
os.makedirs("data/text_files", exist_ok=True)

In [8]:
sample_text = {
    "data/text_files/AI_intro.txt": """
Artificial Intelligence (AI) is transforming industries by enabling machines to perform tasks that typically require human intelligence.

Machine Learning (ML), a subset of AI, allows systems to learn patterns from data and improve over time without being explicitly programmed.

Natural Language Processing (NLP) focuses on helping computers understand and generate human language.

Retrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.
""",

    "data/text_files/python_intro.txt": """
Python is a powerful and beginner-friendly programming language.

It is widely used in web development, data science, machine learning, automation, and artificial intelligence.

Python has simple syntax, making it easy to learn and use.

Popular Python libraries include NumPy, Pandas, TensorFlow, and Flask.
"""
}

for filepath, content in sample_text.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip())

print("2 sample text files created successfully")

2 sample text files created successfully


In [11]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load()

documents

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a powerful and beginner-friendly programming language.\n\nIt is widely used in web development, data science, machine learning, automation, and artificial intelligence.\n\nPython has simple syntax, making it easy to learn and use.\n\nPopular Python libraries include NumPy, Pandas, TensorFlow, and Flask.')]

In [15]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

dir_text = DirectoryLoader(
    "data/text_files/",
    glob="**/*.txt",          # pattern to match .txt files
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)

documents = dir_text.load()

print(documents)             

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a powerful and beginner-friendly programming language.\n\nIt is widely used in web development, data science, machine learning, automation, and artificial intelligence.\n\nPython has simple syntax, making it easy to learn and use.\n\nPopular Python libraries include NumPy, Pandas, TensorFlow, and Flask.'), Document(metadata={'source': 'data/text_files/AI_intro.txt'}, page_content='Artificial Intelligence (AI) is transforming industries by enabling machines to perform tasks that typically require human intelligence.\n\nMachine Learning (ML), a subset of AI, allows systems to learn patterns from data and improve over time without being explicitly programmed.\n\nNatural Language Processing (NLP) focuses on helping computers understand and generate human language.\n\nRetrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.')]


In [21]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

dir_pdf = DirectoryLoader(
    "data/pdf/",
    glob="**/*.pdf",          # match all PDF files
    loader_cls=PyMuPDFLoader, # correct class name (capital P and M)
    show_progress=False
)

documents_pdf = dir_pdf.load()

print(documents_pdf)

[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2025-09-11T18:05:10+05:30', 'source': 'data/pdf/The future of pharmaceuticals-.pdf', 'file_path': 'data/pdf/The future of pharmaceuticals-.pdf', 'total_pages': 21, 'format': 'PDF 1.7', 'title': 'The future of pharmaceuticals: Artificial intelligence in drug discovery and development', 'author': 'Chen Fu', 'subject': 'Journal of Pharmaceutical Analysis, 15 (2025) 101248. doi:10.1016/j.jpha.2025.101248', 'keywords': '', 'moddate': '2025-09-11T18:07:24+05:30', 'trapped': '', 'modDate': "D:20250911180724+05'30'", 'creationDate': "D:20250911180510+05'30'", 'page': 0}, page_content='Review paper\nThe future of pharmaceuticals: Artiﬁcial intelligence in drug discovery\nand development\nChen Fu a, b, Qiuchen Chen a, c, *\na Department of Pharmacology, School of Pharmacy, China Medical University, Shenyang, 110122, China\nb Pharmaceutical Sciences Laboratory Center, School of Pharmacy, C

In [22]:
type(documents_pdf[0])

langchain_core.documents.base.Document

In [23]:
import numpy as np
import uuid

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

from typing import Dict, List, Tuple, Any
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class EmbeddingManager:
    """Handles document embeddings using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: Hugging Face model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. "
                f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed

        Returns:
            NumPy array of embeddings with shape:
            (len(texts), embedding_dim)
        """

        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()

embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 11363.39it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/n6/jqpfr3890bzfbbynf468429c0000gp/T/ipykernel_13050/3420365113.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


In [29]:
import os
import uuid
import numpy as np
import chromadb

from typing import List, Any


class VectorStore:
    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store.
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and store them in vector DB.
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata) if hasattr(doc, "metadata") else {}
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(emb.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )

            print(f"Successfully added {len(documents)} documents")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise


# initialize
vector_store = VectorStore()

vector_store

Vector store initialized. Collection: pdf_documents
Existing documents: 0


In [37]:
# Convert chunk text into embeddings

texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

print("Embedding shape:", embeddings.shape)

NameError: name 'chunks' is not defined

In [ ]:
## convert text into embdding